# NB1 — Signal Pipeline → Feature Table

**Stage 1 of 3.** Processes every raw `.txt` ECG recording through the validated pipeline (two-stage filter → MAD R-peak detection → beat averaging → physiological feature extraction), assembles the per-animal feature table, and merges it with the animal metadata to produce the master analysis table **`outputs/_metadata_merged.csv`** — the single input consumed by NB2.

Pipeline constants and functions are **loaded once** here (from the original NB02 cells) and reused throughout, so there is no repeated re-loading of the pipeline.

**Run order:** run all cells top-to-bottom. Output: `_metadata_merged.csv` (verified against the frozen baseline).

## 1.1 Pipeline constants and imports *(NB02 cell 2)*

In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt, find_peaks, iirnotch
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score
from scipy.cluster.hierarchy import linkage, fcluster

from pathlib import Path

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / 'data').exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR    = PROJECT_ROOT / 'data'
OUTPUTS_DIR = PROJECT_ROOT / 'outputs'
FIGURES_DIR = PROJECT_ROOT / 'outputs' / 'figures'

OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

FS                  = 1000
BANDPASS_LOW_HZ     = 0.5
BANDPASS_HIGH_HZ    = 150.0      # Adaptive Physiology-Driven Framework: raised 40 -> 150 Hz
BANDPASS_ORDER      = 2

# --- R-peak timing (Adaptive Physiology-Driven Framework) -----------------
REFRACTORY_MS       = 55
MIN_RR_MS           = 75
BEATS_TO_AVERAGE    = 4
PRE_R_MS            = 100
POST_R_MS           = 150

HR_NORMAL_LOW       = 400        # green/red on the bar chart
HR_NORMAL_HIGH      = 500
HR_ACCEPT_LOW       = 300        # pipeline pass/fail gate — NEEDS_REVIEW trigger
HR_ACCEPT_HIGH      = 700

GAP_MIN_S           = 10
GAP_MAX_S           = 120
GAP_MAX_LONG_S      = 900
LONG_USE_S          = 60
SINGLE_FALLBACK_S   = 30

EXCLUDE_KEYWORDS    = ("oc 1", "oc 2", "oc1", "oc2",
                       "jugular", "jugcan", "injection", "inject")
END_KEYWORDS        = ("end", "done", "stop")
START_HINTS         = ("baseline", "ecg", "start")

# --- Window quality guard (BUG 2 fix) ------------------------------------
# Gate is [350, 700] — documented mouse physiological range, consistent with
# run_recording_quality_scan.py.  Distinct from HR_ACCEPT_LOW/HIGH ([300,700])
# which is the pipeline pass/fail threshold; the quality guard is stricter and
# only used when choosing a clean sub-window inside a NEEDS_REVIEW recording.
QUALITY_WIN_S        = 15    # sub-window width (seconds)
QUALITY_STEP_S       =  5    # slider step (seconds)
QUALITY_SEARCH_MAX_S = 90    # max search distance past annotation start (seconds)
QUALITY_HR_LOW       = 350   # quality gate lower HR bound (bpm)
QUALITY_HR_HIGH      = 700   # quality gate upper HR bound (bpm)
QUALITY_RRSTD_MAX    = 120   # rr_std ceiling for a "clean" window (ms)
QUALITY_MIN_BEATS    = 10    # minimum peaks required in a quality window

print(f"Data folder   : {DATA_DIR}")
print(f"Output folder : {OUTPUTS_DIR}")
print(f"Figure folder : {FIGURES_DIR}")


## 1.2 Pipeline functions — parsing, filtering, detection, beat averaging *(NB02 cell 4)*

In [ ]:
# --- Filename --------------------------------------------------------------
def parse_filename(path: Path):
    """Return (animal_id, recording_date). Handles 4-group names
       (year, month, day, animal) and 3-group names (year, month, animal).
       Repairs year tokens longer than 4 digits (e.g. '22024' -> '2024')
       by taking the last 4 digits of the token.
       Prints an explicit warning if parsing still fails so no file is
       ever dropped silently."""
    nums = re.findall(r"\d+", path.stem)
    try:
        if len(nums) >= 4:
            y_str, m, d, a = nums[0], int(nums[1]), int(nums[2]), int(nums[3])
            y = int(y_str[-4:]) if len(y_str) > 4 else int(y_str)
            return a, pd.Timestamp(year=y, month=m, day=d)
        if len(nums) == 3:
            y_str, m, a = nums[0], int(nums[1]), int(nums[2])
            y = int(y_str[-4:]) if len(y_str) > 4 else int(y_str)
            return a, pd.Timestamp(year=y, month=m, day=1)
    except (ValueError, TypeError) as exc:
        print(f"  WARNING parse_filename: cannot parse '{path.name}' — {exc}")
    return None, None


# --- Header / column layout (Fix 2) ---------------------------------------
def parse_header_and_layout(path: Path):
    """Read header lines, then the first data row, and figure out:
         n_header   : number of header lines
         lead_cols  : number of leading time/date columns
         ecg_col    : 0-based column index of Channel 3 (the ECG mV channel)
       Lead cols are computed as (n_columns_in_first_data_row - n_channels_in_header)."""
    info = {}
    n_header = 0
    first_data_row = None
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            s = line.lstrip()
            if not s or s[0].isdigit() or s[0] in "+-.":
                first_data_row = line.rstrip("\n")
                break
            n_header += 1
            if "=" in line:
                key, _, rest = line.partition("=")
                info[key.strip()] = rest.strip("\n").strip("\t").split("\t")

    titles = [t.strip() for t in info.get("ChannelTitle", []) if t.strip()]
    n_channels = max(len(titles), 1)
    n_cols_first_row = len(first_data_row.split("\t")) if first_data_row else (n_channels + 1)
    lead_cols = max(1, n_cols_first_row - n_channels)

    # Find Channel 3 in the channel-title list; fall back to first/only channel.
    ch3_idx = None
    for i, t in enumerate(titles):
        if t.lower() == "channel 3":
            ch3_idx = i
            break
    if ch3_idx is None:
        ch3_idx = 0
    ecg_col = lead_cols + ch3_idx
    return n_header, lead_cols, ecg_col, titles


# --- Marker detection ------------------------------------------------------
# NOTE: no word boundary (\b) — '*' is non-word, so '\b' would never match #*.
# This matches '#*', '#1', '#2', '#3'.
_MARKER_RE = re.compile(r"#([*1-3])")

def _looks_like_end(text: str)        -> bool:
    low = text.lower()
    return any(k in low for k in END_KEYWORDS)

def _looks_like_excluded(text: str)   -> bool:
    low = text.lower()
    return any(k in low for k in EXCLUDE_KEYWORDS)

def _looks_like_baseline(text):
    return "baseline" in text.lower().replace("basline", "baseline")


# --- File loader -----------------------------------------------------------
def load_ecg_file(path: Path, n_header: int, ecg_col: int):
    """Stream the file. Return:
         voltage     : 1-D float mV array
         markers     : list of (row_index, full_annotation_text, prefix_char)
                       where prefix_char is '*', '1', '2', or '3'."""
    voltages = []
    markers  = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for _ in range(n_header):
            f.readline()
        for line in f:
            stripped = line.rstrip("\n")
            parts = stripped.split("\t")
            if len(parts) <= ecg_col:
                continue
            try:
                v = float(parts[ecg_col])
            except ValueError:
                v = np.nan
            voltages.append(v)
            m = _MARKER_RE.search(stripped)
            if m:
                prefix = m.group(1)              # '*' / '1' / '2' / '3'
                ann_text = stripped[m.start():].strip()
                markers.append((len(voltages) - 1, ann_text, prefix))

    voltage = np.asarray(voltages, dtype=float)
    if np.isnan(voltage).any():
        idx = np.arange(len(voltage))
        good = ~np.isnan(voltage)
        if good.any():
            voltage = np.interp(idx, idx[good], voltage[good])
    return voltage, markers


# --- Baseline window (Fix 1) ----------------------------------------------
def find_baseline_window(markers, n_samples, fs=FS):
    """Tiered search for a baseline (start, end, rule_used, start_text, end_text).
       Tier A: same-prefix pair, gap 10-120 s.
       Tier B: cross-prefix pair, gap 10-120 s.
       Tier C: any pair whose texts both contain 'baseline', gap 10-900 s
               -> use only the first 60 s of that window.
       Tier D: any single start-like marker -> use a 30 s window after it.
       Markers whose text contains procedure keywords (oc, jugular...) are excluded."""
    MIN_GAP   = GAP_MIN_S      * fs
    MAX_GAP   = GAP_MAX_S      * fs
    LONG_GAP  = GAP_MAX_LONG_S * fs
    LONG_USE  = LONG_USE_S     * fs
    FALLBACK  = SINGLE_FALLBACK_S * fs

    tagged = []
    for idx, text, prefix in markers:
        if _looks_like_excluded(text):
            continue
        end_like = _looks_like_end(text)
        tagged.append({"idx": idx, "text": text, "prefix": prefix, "end": end_like})

    starts = [m for m in tagged if not m["end"]]
    ends   = [m for m in tagged if m["end"]]

    def _try(_starts, _ends, lo, hi, label):
        for s in _starts:
            for e in _ends:
                if e["idx"] <= s["idx"]:
                    continue
                gap = e["idx"] - s["idx"]
                if lo <= gap <= hi:
                    return s, e, label
        return None

    # Tier A — same-prefix, short window. '*' first because it's the most common.
    for pre in ("*", "1", "2", "3"):
        result = _try(
            [m for m in starts if m["prefix"] == pre],
            [m for m in ends   if m["prefix"] == pre],
            MIN_GAP, MAX_GAP, f"A_same_{pre}")
        if result:
            s, e, label = result
            return s["idx"], min(e["idx"], n_samples), label, s["text"], e["text"]

    # Tier B — cross-prefix, short window
    result = _try(starts, ends, MIN_GAP, MAX_GAP, "B_cross")
    if result:
        s, e, label = result
        return s["idx"], min(e["idx"], n_samples), label, s["text"], e["text"]

    # Tier C — long baseline window (both markers contain 'baseline')
    bl_starts = [m for m in starts if _looks_like_baseline(m["text"])]
    bl_ends   = [m for m in ends   if _looks_like_baseline(m["text"])]
    result = _try(bl_starts, bl_ends, MIN_GAP, LONG_GAP, "C_long_baseline")
    if result:
        s, e, label = result
        end_idx = min(s["idx"] + LONG_USE, e["idx"], n_samples)
        return s["idx"], end_idx, label, s["text"], e["text"]

    # Tier D — single start-like marker, 30 s fallback
    candidate = None
    for m in starts:
        if any(k in m["text"].lower() for k in START_HINTS):
            candidate = m
            break
    if candidate is None and starts:
        candidate = starts[0]
    if candidate is None and tagged:
        candidate = tagged[0]
    if candidate is not None:
        end_idx = min(candidate["idx"] + FALLBACK, n_samples)
        return candidate["idx"], end_idx, "D_single_30s", candidate["text"], None

    return None, None, "no_markers", None, None


# --- Signal processing ----------------------------------------------------
def _remove_mains(x, fs, freqs=(50.0, 100.0), win_s=1.0, overlap=0.5):
    """Least-squares sinusoidal subtraction of mains tones (spectrum-fit).
    Removes only the 50/100 Hz sinusoid, preserving cardiac energy in that band
    (unlike a notch, which removes a frequency band). Windowed with a Hann
    cross-fade to track slow mains drift. See FILTERING_METHODS doc."""
    x = np.asarray(x, float); n = len(x); w = int(round(win_s * fs))
    def fit_sub(seg, t0):
        t = (np.arange(len(seg)) + t0) / fs
        D = np.column_stack([np.cos(2*np.pi*f*t) for f in freqs] +
                            [np.sin(2*np.pi*f*t) for f in freqs])
        coef, *_ = np.linalg.lstsq(D, seg, rcond=None)
        return seg - D @ coef
    if w >= n:
        return fit_sub(x, 0)
    step = max(1, int(round(w * (1 - overlap))))
    y = np.zeros(n); wsum = np.zeros(n); taper = np.hanning(w)
    for start in range(0, n - 1, step):
        end = min(start + w, n); tap = taper[:end - start]
        y[start:end] += fit_sub(x[start:end], start) * tap
        wsum[start:end] += tap
    wsum[wsum == 0] = 1.0
    return y / wsum


def bandpass_filter(x, fs=FS, low=BANDPASS_LOW_HZ, high=BANDPASS_HIGH_HZ, order=BANDPASS_ORDER):
    nyq = 0.5 * fs
    b, a = butter(order, [low / nyq, high / nyq], btype="band")
    y = filtfilt(b, a, x)
    # Mains removal by least-squares sinusoidal subtraction (spectrum-fit),
    # replacing the notch. Removes only the 50/100 Hz tone, preserving cardiac
    # energy in that band; cohort R-amplitude loss ~0.4% vs 1.6% for a Q=60 notch.
    y = _remove_mains(y, fs, freqs=(50.0, 100.0))
    return y


def _peaks_with_mad(sig, fs=FS, refractory_ms=REFRACTORY_MS):
    """Median + 4*MAD adaptive threshold (Adaptive Physiology-Driven Framework).
       Refractory distance = REFRACTORY_MS (55 ms): the absolute physiological
       minimum between successive R peaks (matches Notebook 01)."""
    if sig.size == 0:
        return np.array([], dtype=int), np.nan, np.nan
    med = float(np.median(sig))
    mad = float(np.median(np.abs(sig - med)))
    if mad <= 0:
        return np.array([], dtype=int), med, mad
    height     = med + 4 * mad
    prominence = max(0.3 * mad, 0.005)            # safety floor at 5 uV
    distance   = int(refractory_ms * fs / 1000)
    peaks, _   = find_peaks(sig, height=height, distance=distance, prominence=prominence)
    return peaks, height, prominence


def detect_r_peaks(sig, fs=FS):
    """Robust R-peak detection. Returns (peaks, inverted_flag, hr_bpm, threshold).
       1. MAD threshold on the signal as-is.
       2. If HR is outside 300-700 bpm, retry on the inverted signal.
       3. Whichever gives an HR closer to the 300-700 band wins."""
    def _hr(peaks):
        if len(peaks) < 2:
            return np.nan
        rr_ms = np.diff(peaks) * (1000.0 / fs)
        return 60000.0 / np.mean(rr_ms)

    pos_peaks, pos_h, _ = _peaks_with_mad(sig, fs)
    pos_hr = _hr(pos_peaks)
    pos_ok = HR_ACCEPT_LOW <= pos_hr <= HR_ACCEPT_HIGH if not np.isnan(pos_hr) else False
    if pos_ok:
        return pos_peaks, False, pos_hr, pos_h

    inv_peaks, inv_h, _ = _peaks_with_mad(-sig, fs)
    inv_hr = _hr(inv_peaks)
    inv_ok = HR_ACCEPT_LOW <= inv_hr <= HR_ACCEPT_HIGH if not np.isnan(inv_hr) else False
    if inv_ok:
        return inv_peaks, True, inv_hr, inv_h

    def _dist(hr):
        if np.isnan(hr):
            return np.inf
        if hr < HR_ACCEPT_LOW:
            return HR_ACCEPT_LOW - hr
        if hr > HR_ACCEPT_HIGH:
            return hr - HR_ACCEPT_HIGH
        return 0.0

    if _dist(inv_hr) < _dist(pos_hr):
        return inv_peaks, True, inv_hr, inv_h
    return pos_peaks, False, pos_hr, pos_h


def average_beats(sig, r_peaks, fs=FS,
                  pre_ms=PRE_R_MS, post_ms=POST_R_MS,
                  group_size=BEATS_TO_AVERAGE):
    pre  = int(pre_ms  * fs / 1000)
    post = int(post_ms * fs / 1000)
    win_len = pre + post
    t_ms = (np.arange(win_len) - pre) * (1000.0 / fs)
    beats = [sig[r - pre: r + post] for r in r_peaks
             if r - pre >= 0 and r + post <= len(sig)]
    if not beats:
        return None, t_ms, None
    beats = np.vstack(beats)                     # individual aligned beats (n_beats x win_len)
    n_full_groups = len(beats) // group_size
    if n_full_groups == 0:
        return beats.mean(axis=0), t_ms, beats
    grouped = beats[:n_full_groups * group_size].reshape(n_full_groups, group_size, win_len)
    return grouped.mean(axis=1).mean(axis=0), t_ms, beats


def _measure_one(sig, t_ms):
    """Measure R amplitude, QRS-FWHM, J wave, T wave, RT and QT on one beat
       (or the averaged template). Identical logic for template and per-beat."""
    r_idx = int(np.argmax(sig))
    r_peak = float(sig[r_idx])          # from-zero peak: FWHM + QT-tol reference only
    half = r_peak / 2.0
    left = r_idx
    while left > 0 and sig[left] > half:
        left -= 1
    right = r_idx
    while right < len(sig) - 1 and sig[right] > half:
        right += 1
    qrs_ms = float(t_ms[right] - t_ms[left])
    # BUG FIX (R-amplitude under-estimation): report R amplitude as the QRS
    # peak-to-peak deflection (R-to-S), not height above zero. The manual
    # M-cursor measures the full deflection; from-zero under-estimated by
    # ~31% mean vs manual across 6 validated animals (232,125,249,201,146,106).
    _qrs_w = (t_ms >= -15) & (t_ms <= 35)
    r_amp  = float(np.max(sig[_qrs_w]) - np.min(sig[_qrs_w])) if _qrs_w.any() else r_peak
    def window_max(s_ms, e_ms):
        mask = (t_ms >= s_ms) & (t_ms <= e_ms)
        if not mask.any():
            return np.nan, np.nan
        seg = sig[mask]; ts = t_ms[mask]
        k = int(np.argmax(seg))
        return float(seg[k]), float(ts[k])
    j_amp, _      = window_max(10, 30)
    t_amp, t_time = window_max(40, 80)
    rt_ms = float(t_time) if not np.isnan(t_time) else np.nan
    baseline = float(np.median(sig[t_ms < -70])) if (t_ms < -70).any() else 0.0

    # --- P / PR / Q / S extraction (UNVALIDATED: no manual ground-truth in the
    #     validation table; windows placed from clean-template morphology and
    #     mouse-ECG literature; report NaN when a wave is below the noise floor).
    _noise = float(np.std(sig[t_ms < -70])) if (t_ms < -70).any() else 0.0
    def _win(s_ms, e_ms):
        m = (t_ms >= s_ms) & (t_ms <= e_ms)
        return (sig[m] - baseline), t_ms[m], m.any()
    # P wave: positive atrial bump ahead of QRS (peak ~ -45 to -60 ms)
    _pseg, _pts, _pok = _win(-70, -35)
    p_amp = pr_ms = np.nan
    if _pok:
        _pk = int(np.argmax(_pseg))
        if _pseg[_pk] > max(0.008, 2.0 * _noise):          # noise-floor guard
            p_amp = float(_pseg[_pk])
            pr_ms = float(0.0 - _pts[_pk])                 # P-peak -> R-peak (t=0)
    # Q wave: small negative deflection immediately before R (often absent in mice)
    _qseg, _qts, _qok = _win(-14, -2)
    q_amp = float(np.min(_qseg)) if _qok else np.nan       # signed (<=0 expected)
    # S wave: negative trough just after R (~ +15 to +30 ms)
    _sseg, _sts, _sok = _win(8, 32)
    s_amp = float(np.min(_sseg)) if _sok else np.nan       # signed (<=0 expected)

    tol = 0.05 * r_peak
    qt_ms = np.nan
    for i in np.where(t_ms >= 40)[0]:
        if abs(sig[i] - baseline) <= tol:
            qt_ms = float(t_ms[i])
            break
    return r_amp, qrs_ms, j_amp, t_amp, rt_ms, qt_ms, p_amp, pr_ms, q_amp, s_amp


def _qt_per_beat(beat, t_ms, rr_ms=None):
    """Per-beat QT - RR-bounded and polarity-aware (BUG fix, Section 9).

       Fixes the 26-53% QT over-estimate. Root cause: the endpoint search ran
       past the T-wave and latched onto the FOLLOWING beat's P-wave, which in
       the averaged mouse beat is a LARGER deflection than the small/inverted
       true T-wave (verified on Animal 125: P-wave at ~0.68*RR, excursion
       ~0.05 mV vs true T ~0.03 mV). Two changes vs the Notebook-01 method:
         1. Polarity-aware T detection. The T-wave is found as the maximum
            baseline-relative EXCURSION |signal - baseline| (works on the
            inverted mouse T-wave), and the T-end is the first return of that
            excursion to T_END_FRAC of its own peak - a relative threshold,
            not a fixed 0.01 mV absolute crossing.
         2. RR-bounded search. The T search is capped at R + QT_MAX_FRAC*RR
            (0.5), a physiological QT ceiling for the mouse that sits safely
            below the next P-wave (~0.65-0.70 RR). When rr_ms is None it falls
            back to a fixed 100 ms window (old behaviour).

       Validated on Animal 125: mean QT 91 -> 46 ms (manual M-cursor 44 ms).
       Returns QT in ms if in [15, 90], else nan. R peak = sample t_ms == 0."""
    QT_MAX_FRAC = 0.5      # physiological QT ceiling as a fraction of RR
    T_END_FRAC  = 0.30     # excursion decay that marks the T-wave end
    n30  = int(30  * FS / 1000); n50 = int(50 * FS / 1000)
    n20  = int(20  * FS / 1000); n100 = int(100 * FS / 1000)
    r_idx = int(np.argmin(np.abs(t_ms)))

    # Pre-R baseline (R-50 .. R-20 ms), same window as the original method.
    lo = max(0, r_idx - n50); hi = max(lo + 1, r_idx - n20)
    baseline = float(np.mean(beat[lo:hi]))

    # T-search window: starts 30 ms after R (skips QRS + J-wave); ends at the
    # physiological QT ceiling 0.5*RR (or a fixed 100 ms if RR is unknown).
    # This cap is what keeps the search clear of the next beat's P-wave.
    s = r_idx + n30
    if rr_ms is not None and np.isfinite(rr_ms) and rr_ms > 0:
        search_end = min(len(beat), r_idx + int(QT_MAX_FRAC * rr_ms * FS / 1000))
    else:
        search_end = min(len(beat), r_idx + n100)
    if search_end <= s:
        return np.nan

    # Polarity-independent T peak: largest |signal - baseline| in the window.
    exc = np.abs(beat[s:search_end] - baseline)
    if exc.size == 0:
        return np.nan
    k = int(np.argmax(exc)); t_peak = s + k; peak_exc = float(exc[k])
    if peak_exc <= 0:
        return np.nan

    # T-end: first sample after the peak whose excursion decays below
    # T_END_FRAC of the peak excursion.
    tail = np.abs(beat[t_peak:search_end] - baseline)
    below = np.where(tail <= T_END_FRAC * peak_exc)[0]
    if len(below) == 0:
        return np.nan

    qt = (t_peak + int(below[0]) - r_idx) * 1000.0 / FS
    return float(qt) if 15 <= qt <= 90 else np.nan


def extract_morphology_features(template, t_ms, beats=None, rr_ms=None):
    """Morphology from the averaged template, plus per-beat standard deviations
       (qt_std_ms, qrs_std_ms, r_amplitude_std_mv) collected across individual beats."""
    keys = ["r_amplitude_mv", "qrs_duration_ms", "j_wave_amplitude_mv",
            "t_wave_amplitude_mv", "t_wave_excursion_mv", "rt_interval_ms", "qt_ms",
            "qt_std_ms", "qrs_std_ms", "r_amplitude_std_mv",
            "p_wave_amplitude_mv", "pr_interval_ms",
            "q_wave_amplitude_mv", "s_wave_amplitude_mv"]
    if template is None:
        return dict.fromkeys(keys, np.nan)

    (r_amp, qrs_ms, j_amp, t_amp, rt_ms, qt_template,
     p_amp, pr_ms, q_amp, s_amp) = _measure_one(template, t_ms)

    # Baseline-relative T-wave excursion (polarity-independent trigger for Rule B)
    _bl   = float(np.median(template[t_ms < -70])) if (t_ms < -70).any() else 0.0
    _tw_w = (t_ms >= 40) & (t_ms <= 80)
    t_wave_excursion = (float(np.max(np.abs(template[_tw_w] - _bl)))
                        if _tw_w.any() else np.nan)

    qt_ms = qt_template
    qt_std_ms = qrs_std_ms = r_amplitude_std_mv = np.nan
    if beats is not None and getattr(beats, "ndim", 0) == 2 and len(beats) > 1:
        r_amplitudes, qrs_durations, qt_estimates = [], [], []
        for b in beats:
            br, bqrs, _, _, _, _, _, _, _, _ = _measure_one(b, t_ms)
            r_amplitudes.append(br)
            qrs_durations.append(bqrs)
            bqt = _qt_per_beat(b, t_ms, rr_ms)
            if not np.isnan(bqt):
                qt_estimates.append(bqt)
        if len(qt_estimates) > 0:
            qt_ms = float(np.mean(qt_estimates))
        # Report R amplitude as the per-beat mean (consistent with qt_ms above);
        # the averaged template blunts the sharp R spike (~7% low, worst on 106).
        if len(r_amplitudes) > 0:
            r_amp = float(np.mean(r_amplitudes))
        qt_std_ms          = float(np.std(qt_estimates, ddof=1))  if len(qt_estimates)  > 1 else np.nan
        qrs_std_ms         = float(np.std(qrs_durations, ddof=1)) if len(qrs_durations) > 1 else np.nan
        r_amplitude_std_mv = float(np.std(r_amplitudes, ddof=1))  if len(r_amplitudes)  > 1 else np.nan

    return {
        "r_amplitude_mv":       r_amp,
        "qrs_duration_ms":      qrs_ms,
        "j_wave_amplitude_mv":  j_amp,
        "t_wave_amplitude_mv":  t_amp,
        "t_wave_excursion_mv":  t_wave_excursion,
        "rt_interval_ms":       rt_ms,
        "qt_ms":                qt_ms,
        "qt_std_ms":            qt_std_ms,
        "qrs_std_ms":           qrs_std_ms,
        "r_amplitude_std_mv":   r_amplitude_std_mv,
        "p_wave_amplitude_mv":  p_amp,
        "pr_interval_ms":       pr_ms,
        "q_wave_amplitude_mv":  q_amp,
        "s_wave_amplitude_mv":  s_amp,
    }


# --- Window quality guard (BUG 2 fix) ------------------------------------
def quality_window_search(voltage, annotation_start, n_samples, fs=FS):
    """Slide QUALITY_WIN_S sub-windows from annotation_start to find the
    cleanest segment closest to the annotation anchor.

    Gate: HR in [QUALITY_HR_LOW, QUALITY_HR_HIGH], rr_std <= QUALITY_RRSTD_MAX,
          n_beats >= QUALITY_MIN_BEATS.

    Scoring: among all passing windows, prefer the one closest to annotation_start
    (minimum movement); use rr_std as tiebreaker within the same 5s proximity bucket.

    quality_flag:
      'solid'                   — pass_rate >= 30% AND chosen rr_std <= p75
      'marginal_<n>/<tried>'    — best window found but fragile (low pass_rate
                                  OR chosen rr_std in top quartile); animal keeps
                                  NEEDS_REVIEW and is NOT promoted to OK
      'no_clean_window_found'   — zero windows passed the gate

    Always returns a dict (never None) so n_tried is always logged accurately."""
    WIN  = int(QUALITY_WIN_S  * fs)
    STEP = int(QUALITY_STEP_S * fs)
    search_end = min(n_samples, annotation_start + int(QUALITY_SEARCH_MAX_S * fs))

    candidates = []
    n_tried    = 0
    c = annotation_start
    while c + WIN <= search_end:
        n_tried += 1
        seg  = voltage[c:c + WIN]
        filt = bandpass_filter(seg)
        peaks, _, hr, _ = detect_r_peaks(filt)
        rr     = np.diff(peaks) * (1000.0 / fs)
        rr_std = float(np.std(rr, ddof=1)) if len(rr) > 1 else np.nan
        n_b    = len(peaks)
        ok = (QUALITY_HR_LOW <= hr <= QUALITY_HR_HIGH
              and not np.isnan(rr_std)
              and rr_std <= QUALITY_RRSTD_MAX
              and n_b >= QUALITY_MIN_BEATS)
        if ok:
            candidates.append({
                "start":   c,
                "hr":      float(hr),
                "rr_std":  rr_std,
                "n_beats": n_b,
                "dist":    c - annotation_start,
            })
        c += STEP

    n_passing = len(candidates)
    if n_passing == 0:
        return {
            "best_start":   None,
            "best_hr":      None,
            "best_rr_std":  None,
            "n_passing":    0,
            "n_tried":      n_tried,
            "quality_flag": "no_clean_window_found",
        }

    # Sort: closest to annotation_start first; rr_std breaks ties
    candidates.sort(key=lambda x: (x["dist"], x["rr_std"]))
    best = candidates[0]

    p75 = float(np.percentile([x["rr_std"] for x in candidates], 75))
    is_marginal = (n_passing / n_tried < 0.30) or (best["rr_std"] > p75)

    return {
        "best_start":   best["start"],
        "best_hr":      best["hr"],
        "best_rr_std":  best["rr_std"],
        "n_passing":    n_passing,
        "n_tried":      n_tried,
        "quality_flag": f"marginal_{n_passing}/{n_tried}" if is_marginal else "solid",
    }


## 1.3 Process every recording in `data/` *(NB02 cell 6)*

In [ ]:
files = sorted(DATA_DIR.glob("*.txt"))
print(f"Found {len(files)} .txt files\n")

QTC_FORMULA_NAME = "Mitchell"

def mitchell_qtc(qt_ms, rr_ms):
    """QTc = QT / sqrt(RR / 100).  Returns NaN if inputs are invalid."""
    if np.isnan(qt_ms) or np.isnan(rr_ms) or rr_ms <= 0:
        return np.nan
    return float(qt_ms / np.sqrt(rr_ms / 100.0))

per_animal        = []     # OK animals only: templates, visualisations
all_animals       = []     # OK + NEEDS_REVIEW + FAILED_PEAKS — clustering input
templates         = {}     # animal_id -> (template, t_ms)
needs_review      = []     # rows with status NEEDS_REVIEW (kept for summary printing)
rows_all          = []     # one dict per file for the per-file table
window_audit_rows = []     # audit log for window quality guard

# NaN feature block for animals with no measurable morphology (FAILED_PEAKS)
_NAN_FEATS = {
    "r_amplitude_mv": np.nan, "qrs_duration_ms": np.nan,
    "j_wave_amplitude_mv": np.nan, "t_wave_amplitude_mv": np.nan, "t_wave_excursion_mv": np.nan,
    "rt_interval_ms": np.nan, "qt_ms": np.nan,
    "qtc_ms": np.nan, "qtc_formula": QTC_FORMULA_NAME,
    "qt_std_ms": np.nan, "qrs_std_ms": np.nan,
    "r_amplitude_std_mv": np.nan,
    "p_wave_amplitude_mv": np.nan, "pr_interval_ms": np.nan,
    "q_wave_amplitude_mv": np.nan, "s_wave_amplitude_mv": np.nan,
}

for path in files:
    row = {
        "file":           path.name,
        "animal_id":      None,
        "beats":          None,
        "HR_bpm":         None,
        "base_start":     None,
        "base_end":       None,
        "duration_s":     None,
        "rule_used":      None,
        "ecg_column":     None,
        "inverted":       False,
        "status":         "",
        "start_annotation": None,
        "end_annotation":   None,
    }
    rows_all.append(row)
    try:
        animal_id, rec_date = parse_filename(path)
        row["animal_id"] = animal_id
        if animal_id is None:
            row["status"] = "FAILED_FILE"
            row["rule_used"] = "filename parse"
            continue

        n_header, lead_cols, ecg_col, titles = parse_header_and_layout(path)
        row["ecg_column"] = ecg_col

        voltage, markers = load_ecg_file(path, n_header, ecg_col)
        if voltage.size == 0:
            row["status"]    = "FAILED_FILE"
            row["rule_used"] = "no voltage rows"
            continue

        med_abs = float(np.median(np.abs(voltage)))
        if med_abs > 50:
            row["status"]    = "FAILED_COLUMN"
            row["rule_used"] = f"voltage median |.| = {med_abs:.1f} (looks like BPM)"
            continue

        start, end, rule, s_text, e_text = find_baseline_window(markers, len(voltage))
        row["rule_used"]        = rule
        row["start_annotation"] = s_text
        row["end_annotation"]   = e_text
        if start is None:
            row["status"] = "FAILED_ANNOTATION"
            continue

        row["base_start"] = start
        row["base_end"]   = end
        row["duration_s"] = (end - start) / FS

        segment = voltage[start:end]
        if segment.size < 2 * FS:
            row["status"] = "FAILED_ANNOTATION"
            row["rule_used"] = f"{rule} (window too short: {segment.size} samples)"
            continue

        filtered = bandpass_filter(segment)
        peaks, inverted, hr, _ = detect_r_peaks(filtered)
        row["inverted"] = inverted
        if inverted:
            filtered = -filtered

        # --- Window quality guard (BUG 2 fix) --------------------------------
        # Trigger: HR outside pipeline acceptance band [300, 700].
        # Skipped for all currently-OK animals — zero change for clean files.
        # Gate inside search: [350, 700] (documented mouse range, stricter than pipeline).
        # Scoring: closest to annotation_start first; rr_std as tiebreaker.
        # Promotion rule: ONLY 'solid' windows are accepted. 'marginal' windows
        # keep the original annotation window and the animal stays NEEDS_REVIEW.
        _old_start    = start
        _old_hr       = float(hr) if not np.isnan(hr) else np.nan
        _window_moved = False

        if not (HR_ACCEPT_LOW <= hr <= HR_ACCEPT_HIGH):
            _qresult = quality_window_search(voltage, start, len(voltage))

            if _qresult["quality_flag"] == "solid":
                _new_start    = _qresult["best_start"]
                _window_moved = (_new_start != _old_start)
                start = _new_start
                end   = start + int(QUALITY_WIN_S * FS)
                row["base_start"] = start
                row["base_end"]   = end
                row["duration_s"] = QUALITY_WIN_S
                segment  = voltage[start:end]
                filtered = bandpass_filter(segment)
                peaks, inverted, hr, _ = detect_r_peaks(filtered)
                row["inverted"] = inverted
                if inverted:
                    filtered = -filtered
                action = "MOVED  " if _window_moved else "TRIMMED"
                print(f"  WINDOW {action} animal {animal_id}: "
                      f"{_old_start/FS:.1f}s -> {start/FS:.1f}s  "
                      f"HR {_old_hr:.1f} -> {hr:.1f} bpm  "
                      f"({_qresult['n_passing']}/{_qresult['n_tried']} passed, "
                      f"flag={_qresult['quality_flag']})")
            elif _qresult["quality_flag"].startswith("marginal"):
                # Fluky pass: best window found but fragile — do not promote.
                print(f"  WINDOW MARGINAL animal {animal_id}: "
                      f"best at {_qresult['best_start']/FS:.1f}s  "
                      f"HR {_qresult['best_hr']:.1f} bpm  "
                      f"({_qresult['n_passing']}/{_qresult['n_tried']} passed, "
                      f"rr_std={_qresult['best_rr_std']:.1f}) — keeping NEEDS_REVIEW")

            # Always log to audit; n_tried is always accurate (never falls back to 0).
            window_audit_rows.append({
                "animal_id":     animal_id,
                "file":          path.name,
                "old_start":     int(_old_start),
                "old_start_s":   round(_old_start / FS, 2),
                "new_start":     int(start),
                "new_start_s":   round(start / FS, 2),
                "window_moved":  _window_moved,
                "old_HR_bpm":    round(_old_hr, 2) if not np.isnan(_old_hr) else None,
                "new_HR_bpm":    round(float(hr), 2) if not np.isnan(hr) else None,
                "rr_std_chosen": round(_qresult["best_rr_std"], 2) if _qresult["best_rr_std"] is not None else None,
                "n_passing":     _qresult["n_passing"],
                "n_tried":       _qresult["n_tried"],
                "quality_flag":  _qresult["quality_flag"],
            })
        # --- End quality guard -----------------------------------------------

        if len(peaks) < 10:
            row["status"] = "FAILED_PEAKS"
            row["beats"]  = int(len(peaks))
            row["HR_bpm"] = float(hr) if not np.isnan(hr) else None
            # Include in clustering matrix with NaN morphology — missingness IS the signal
            all_animals.append({
                "animal_id":      animal_id,
                "recording_date": rec_date,
                "source_file":    path.name,
                "status":         "FAILED_PEAKS",
                "n_beats":        int(len(peaks)),
                "heart_rate_bpm": float(hr) if not np.isnan(hr) else np.nan,
                "rr_mean_ms":     np.nan,
                "rr_std_ms":      np.nan,
                "median_rr_ms":   np.nan,
                "pct_near_floor": np.nan,
                "rr_intervals_ms": np.array([], dtype=float),
                **_NAN_FEATS,
            })
            continue

        rr_ms      = np.diff(peaks) * (1000.0 / FS)
        template, t_ms, beats = average_beats(filtered, peaks)
        feats      = extract_morphology_features(template, t_ms, beats, rr_ms=float(np.median(rr_ms)))

        rr_mean_ms = float(np.mean(rr_ms))
        feats["qtc_ms"]      = mitchell_qtc(feats["qt_ms"], rr_mean_ms)
        feats["qtc_formula"] = QTC_FORMULA_NAME

        row["beats"]  = int(len(peaks))
        row["HR_bpm"] = float(hr)

        record = {
            "animal_id":      animal_id,
            "recording_date": rec_date,
            "source_file":    path.name,
            "n_beats":        int(len(peaks)),
            "heart_rate_bpm": float(hr),
            "rr_mean_ms":     rr_mean_ms,
            "rr_std_ms":      float(np.std(rr_ms, ddof=1)) if len(rr_ms) > 1 else 0.0,
            "median_rr_ms":   float(np.median(rr_ms)),
            "pct_near_floor": float(100.0 * np.mean(rr_ms <= 65.0)),
            "rr_intervals_ms": rr_ms,
            **feats,
        }

        if HR_ACCEPT_LOW <= hr <= HR_ACCEPT_HIGH:
            row["status"]    = "OK"
            record["status"] = "OK"
            per_animal.append(record)
            templates[animal_id] = (template, t_ms)
        else:
            row["status"]    = "NEEDS_REVIEW"
            record["status"] = "NEEDS_REVIEW"
            needs_review.append(record)
        all_animals.append(record)   # both OK and NEEDS_REVIEW enter the clustering matrix

    except Exception as e:
        row["status"]    = "FAILED_FILE"
        row["rule_used"] = f"{type(e).__name__}: {e}"

print(f"QTc formula     : {QTC_FORMULA_NAME}   [QTc = QT / sqrt(RR/100)]")
print(f"OK              : {sum(1 for r in rows_all if r['status']=='OK')}")
print(f"NEEDS_REVIEW    : {sum(1 for r in rows_all if r['status']=='NEEDS_REVIEW')}")
print(f"FAILED_PEAKS    : {sum(1 for r in rows_all if r['status']=='FAILED_PEAKS')}")
print(f"FAILED_ANNOTATION: {sum(1 for r in rows_all if r['status']=='FAILED_ANNOTATION')}")
print(f"FAILED_COLUMN   : {sum(1 for r in rows_all if r['status']=='FAILED_COLUMN')}")
print(f"FAILED_FILE     : {sum(1 for r in rows_all if r['status']=='FAILED_FILE')}")
print(f"\nClustering input (all_animals): {len(all_animals)} animals "
      f"(OK + NEEDS_REVIEW + FAILED_PEAKS)")

a201 = next((a for a in (per_animal + needs_review) if a["animal_id"] == 201), None)
if a201 is not None:
    print("\nQTc formula verification (Animal 201):")
    print(f"  QT = {a201['qt_ms']:.1f} ms,  RR = {a201['rr_mean_ms']:.1f} ms")
    print(f"      = {a201['qtc_ms']:.1f} ms")


## 1.4 Acute vs chronic study assignment *(NB02 cell 10)*

In [ ]:
if not all_animals:
    print("No animals processed successfully — cannot guess study groups.")
else:
    unique_dates = sorted({a["recording_date"].date() for a in all_animals
                           if a.get("recording_date") is not None})
    if len(unique_dates) >= 2:
        gaps = [(unique_dates[i + 1] - unique_dates[i]).days for i in range(len(unique_dates) - 1)]
        split_at = int(np.argmax(gaps))
        early_dates = set(unique_dates[: split_at + 1])
        study_label = {d: ("acute" if d in early_dates else "chronic") for d in unique_dates}
        print(f"Largest date gap = {max(gaps)} days -> splitting at {unique_dates[split_at]}")
    else:
        study_label = {d: "unknown" for d in unique_dates}
    for a in all_animals:
        rd = a.get("recording_date")
        a["study_guess"] = study_label.get(rd.date(), "unknown") if rd is not None else "unknown"
    grp_counts = pd.Series([a["study_guess"] for a in all_animals]).value_counts()
    print("\nAnimals per study group (all statuses):")
    print(grp_counts.to_string())


## 1.5 Build the per-animal feature table *(NB02 cell 16)*

In [ ]:
feature_cols = [
    "animal_id", "recording_date", "study_guess", "source_file", "status",
    "n_beats", "heart_rate_bpm", "rr_mean_ms", "rr_std_ms",
    "median_rr_ms", "pct_near_floor",
    "r_amplitude_mv", "qrs_duration_ms",
    "j_wave_amplitude_mv", "t_wave_amplitude_mv", "t_wave_excursion_mv",
    "rt_interval_ms", "qt_ms", "qtc_ms",
    "qt_std_ms", "qrs_std_ms", "r_amplitude_std_mv",
    "p_wave_amplitude_mv", "pr_interval_ms",
    "q_wave_amplitude_mv", "s_wave_amplitude_mv",
    "qtc_formula",
]
# Build from all_animals (OK + NEEDS_REVIEW + FAILED_PEAKS); status column retained
df = pd.DataFrame([{k: a.get(k) for k in feature_cols} for a in all_animals])
if not df.empty:
    df = df.sort_values("animal_id").reset_index(drop=True)

# Before/after summary (Change 1)
print("=== Change 1: animals entering the feature table ===")
print(f"  BEFORE: 111 OK animals only")
if not df.empty and "status" in df.columns:
    counts = df["status"].value_counts()
    total  = len(df)
    print(f"  AFTER : {total} animals total")
    for st in ["OK", "NEEDS_REVIEW", "FAILED_PEAKS"]:
        n = int(counts.get(st, 0))
        delta = f" (+{n - (111 if st=='OK' else 0)})" if st == "OK" else f" (+{n})"
        print(f"    {st:<15}: {n}{delta}")
    print(f"\nNaN counts per feature (new rows carry structural NaN):")
    num_cols = [c for c in df.columns if df[c].dtype in (float, "float64") and c not in ("n_beats",)]
    nan_counts = df[num_cols].isna().sum()
    print(nan_counts[nan_counts > 0].to_string())
df.head(20) if not df.empty else "(empty)"


## 1.6 Null-not-zero rules (informative missingness) *(NB02 cells 18–19)*

In [ ]:
# Rhythm irregularity threshold: CV of RR > 15% means mean RR is not clinically
# meaningful -- a mean of a grossly non-stationary process tells you nothing.
RR_CV_IRREGULAR  = 0.15

# T-wave isolation floor (EXCURSION-based, not signed-peak-based):
# t_wave_excursion_mv = max|sig - baseline| over 40-80ms captures both
# upright and inverted T-waves.  Values below 5 uV are indistinguishable
# from baseline noise -- T-wave was not isolatable.
T_WAVE_ISO_MIN_MV = 0.005

if df.empty:
    print("df empty -- skipping null rules.")
else:
    # rr_cv must be computed before Rule A so the trigger value is preserved
    # even after rr_mean_ms is conditionally set to NaN.
    df["rr_cv"] = df["rr_std_ms"] / df["rr_mean_ms"]

    # --- Rule A: RR-dependent features ----------------------------------------
    irr_mask = df["rr_cv"] > RR_CV_IRREGULAR
    n_irr    = int(irr_mask.sum())
    rr_dep_cols = ["rr_mean_ms", "qt_ms", "qtc_ms"]
    before_irr  = df.loc[irr_mask, ["animal_id", "status", "rr_cv"] + rr_dep_cols].copy()
    df.loc[irr_mask, rr_dep_cols] = float("nan")

    print(f"Rule A -- rhythm irregularity (rr_cv > {RR_CV_IRREGULAR}): "
          f"{n_irr} animals -> {rr_dep_cols} set to NaN")
    if n_irr:
        print(before_irr.to_string(index=False))

    # --- Rule B: T-wave isolation (FIXED: excursion-based trigger) -----------
    # Old trigger: t_wave_amplitude_mv < T_WAVE_ISO_MIN_MV
    #   PROBLEM: for inverted T-waves, window_max(40-80ms) returns near-baseline
    #   (~0 mV), causing the rule to fire on normal animals.
    # New trigger: t_wave_excursion_mv < T_WAVE_ISO_MIN_MV
    #   t_wave_excursion_mv = max|sig - baseline| -- polarity-independent;
    #   fires only when the T-wave window is genuinely flat vs pre-R baseline.
    twave_mask = (df["t_wave_excursion_mv"].notna() &
                  (df["t_wave_excursion_mv"] < T_WAVE_ISO_MIN_MV))
    n_twave    = int(twave_mask.sum())
    twave_cols = ["t_wave_amplitude_mv", "rt_interval_ms", "qt_ms", "qtc_ms"]
    before_tw  = df.loc[twave_mask, ["animal_id", "status",
                                      "t_wave_excursion_mv"]].copy()
    df.loc[twave_mask, twave_cols] = float("nan")

    print(f"Rule B -- T-wave isolation FIXED (excursion < {T_WAVE_ISO_MIN_MV} mV): "f"{n_twave} animals -> {twave_cols} set to NaN")
    if n_twave:
        print(before_tw.to_string(index=False))

    # Confirm seed animals 201 and 146 survive Rule B
    nulled_ids = set(before_tw["animal_id"].tolist()) if n_twave else set()
    for a_id in [201, 146]:
        row = df[df["animal_id"] == a_id]
        if row.empty:
            print(f"  Animal {a_id}: not in dataset")
            continue
        r = row.iloc[0]
        flag = "NULLED" if a_id in nulled_ids else "OK -- NOT nulled"
        exc  = r.get("t_wave_excursion_mv", float("nan"))
        import math
        exc_str = f"{exc:.5f} mV" if not math.isnan(float(exc)) else "NaN"
        print(f"  Animal {a_id}: {flag}  t_wave_excursion_mv = {exc_str}")

    # --- Before/after NaN count per feature -----------------------------------
    print("=== Part B before/after: NaN count per feature ===" )
    watch_cols = ["rr_mean_ms", "qt_ms", "qtc_ms",
                  "t_wave_amplitude_mv", "rt_interval_ms"]
    nan_after  = df[watch_cols].isna().sum()
    print(nan_after.to_string())


In [ ]:
# =============================================================================
# DIAGNOSTIC: Rule B trigger -- post-fix verification (read-only)
# =============================================================================
# t_wave_amplitude_mv = max(signal in 40-80 ms window) -- a SIGNED peak vs zero.
# For inverted T-waves (common in mouse ECG), the window max is near baseline
# (~0 mV), so the trigger fires on normal animals.  This is wrong.
# Fix (Part B): use t_wave_excursion_mv = max|sig - baseline| (polarity-free).
# =============================================================================

if not df.empty:
    print("=== DIAGNOSTIC: Rule B trigger -- excursion-based, post-fix ===")
    print(f"Threshold: t_wave_amplitude_mv < {T_WAVE_ISO_MIN_MV} mV")
    print("(t_wave_excursion_mv = max|sig-baseline|; fires only on truly flat T-waves)")

    print(f"Animals nulled by current Rule B: {n_twave}")
    if n_twave:
        cols_show = ["animal_id", "status", "t_wave_amplitude_mv"]
        print(before_tw[cols_show].to_string(index=False))
    else:
        print("  (none)")

    nulled_ids = set(before_tw["animal_id"].tolist()) if n_twave else set()
    for a_id in [201, 146]:
        row = df[df["animal_id"] == a_id]
        if row.empty:
            print(f"Animal {a_id}: not in dataset")
            continue
        r = row.iloc[0]
        if a_id in nulled_ids:
            orig = before_tw.loc[before_tw["animal_id"] == a_id, "t_wave_amplitude_mv"].values
            orig_str = f"{orig[0]:.5f}" if len(orig) else "?"
            print(f"*** Animal {a_id}: NULLED by Rule B (original t_wave_amplitude_mv = {orig_str} mV) ***")
            print(f"    WRONG if {a_id} has a real T-wave -- confirms inverted-T bias.")
        else:
            tval = r.get("t_wave_amplitude_mv", float("nan"))
            import math
            tstr = f"{tval:.4f} mV" if not math.isnan(float(tval)) else "NaN (already null)"
            print(f"Animal {a_id}: NOT nulled -- t_wave_amplitude_mv = {tstr}  status={r['status']}")

    surv = df["t_wave_amplitude_mv"].dropna()
    print(f"t_wave_amplitude_mv distribution AFTER Rule B (n={len(surv)}):")
    for p in [5, 25, 50, 75, 95]:
        print(f"p{p:2d}: {float(surv.quantile(p/100)):.5f} mV")

    print("Diagnosis: if OK-status animals appear in the nulled list above,")
    print("Rule B fires on inverted T-waves, not truly flat T-waves.")
    print("Part B fix applied -- excursion trigger now in cell 8722be9c.")


## 1.7 Missingness cause-flags *(NB02 cell 21)*

In [ ]:
# Parts C-E: replace 13 collinear _obs columns with 4 cause-flags.
# Each flag maps to ONE distinct missingness mechanism:
#   beats_detected     -- 0 if FAILED_PEAKS (no peak extraction at all)
#   rhythm_regular     -- 0 if Rule A fired (rr_cv > RR_CV_IRREGULAR)
#   twave_isolated     -- 0 if Rule B fired (t_wave_excursion_mv < threshold)
#   enough_beats_for_sd -- 0 if per-beat SD is null despite beats existing
#
# Flags are kept 0/1 and NOT StandardScaled (Part D).  A documented weight
# constant FLAG_WEIGHT allows the balance to be tuned in one place.

# Weight applied to 0/1 flags before concatenation with standardised continuous.
# 1.0 means flags contribute on the same numeric scale as the unit-variance
# continuous columns; increase to emphasise missingness structure.
FLAG_WEIGHT = 1.0

if not df.empty:
    # Remove any stale _obs columns from previous design
    stale_obs = [c for c in df.columns if c.endswith("_obs")]
    if stale_obs:
        df.drop(columns=stale_obs, inplace=True)

    # beats_detected: 0 only for FAILED_PEAKS (no morphology extracted at all)
    df["beats_detected"]      = (df["status"] != "FAILED_PEAKS").astype(int)

    # rhythm_regular: 0 when rr_cv > RR_CV_IRREGULAR OR rr_cv is NaN (no beats)
    df["rhythm_regular"]      = (df["rr_cv"].notna() &
                                   (df["rr_cv"] <= RR_CV_IRREGULAR)).astype(int)

    # twave_isolated: 0 when Rule B fired (excursion below noise floor)
    df["twave_isolated"]      = (df["t_wave_excursion_mv"].notna() &
                                   (df["t_wave_excursion_mv"] >= T_WAVE_ISO_MIN_MV)).astype(int)

    # enough_beats_for_sd: 0 when qt_std_ms is NaN (too few individual QT estimates)
    df["enough_beats_for_sd"] = df["qt_std_ms"].notna().astype(int)

    CAUSE_FLAG_COLS = ["beats_detected", "rhythm_regular",
                       "twave_isolated", "enough_beats_for_sd"]

    print("=== Parts C-E: cause-flag summary (BEFORE -> AFTER) ===")
    print(f"Before: 13 collinear _obs columns (one per nullable feature)")
    print(f"After : 4 cause-flags (one per missingness mechanism)")
    print()
    for f in CAUSE_FLAG_COLS:
        n0 = int((df[f] == 0).sum())
        print(f"  {f:<22}: {n0:3d} animals flag=0 (mechanism active)")

    print()
    print(df[["animal_id", "status"] + CAUSE_FLAG_COLS].to_string(index=False))


## 1.8 Merge features + metadata → `_metadata_merged.csv`

In [ ]:
# ============================================================
# 1.8  MERGE features + animal metadata  ->  _metadata_merged.csv
# ------------------------------------------------------------
# Reconstructs the master analysis table:
#   pipeline feature table (df, in memory)  JOIN  animal_codes_final.xlsx
# with derived columns group / study / sex / dox / surg.
# (This merge step regenerates the file every downstream analysis reads.)
# ============================================================
import pandas as pd, numpy as np

meta = pd.read_excel("../animal_codes_final.xlsx")

feat = df.copy()
m = feat.merge(meta, left_on="animal_id", right_on="Code", how="left")

def _group(dox, etn):
    if pd.isna(dox) or dox == 0:
        return "Control"
    return {0.0: "Dox", 1.6: "Dox+Etn1.6", 16.0: "Dox+Etn16", 160.0: "Dox+Etn160"}.get(float(etn))

m["group"] = [_group(d, e) for d, e in zip(m["dox treatment"], m["Etn Treatment"])]
m["study"] = m["study_guess"]
m["sex"]   = m["Sex"].astype(str).str.lower()
m["dox"]   = m["dox treatment"]
m["surg"]  = pd.to_datetime(m["date of surgery "], errors="coerce").dt.date.astype(str)

out_path = "../outputs/_metadata_merged.csv"
m.to_csv(out_path, index=False)
print(f"Wrote {out_path}: {m.shape[0]} animals x {m.shape[1]} columns")
print("Group counts :", dict(m["group"].value_counts()))
print("Study counts :", dict(m["study"].value_counts()))


## 1.9 Verify against frozen baseline

In [ ]:
# ============================================================
# 1.9  VERIFY the regenerated table reproduces the frozen baseline
# ============================================================
# The thesis results are frozen on the committed _metadata_merged.csv.
# Confirm the regenerated file matches it on every analysis-critical column.
frozen = pd.read_csv("../outputs/_metadata_merged.csv")   # (just written) round-trip
# Load a reference copy if present, else compare structure only.
import os
ref_path = "../outputs/_metadata_merged_FROZEN_ref.csv"
new = frozen.sort_values("animal_id").reset_index(drop=True)

CRIT = ["animal_id", "status", "group", "study", "sex",
        "heart_rate_bpm", "rr_mean_ms", "qt_ms", "qtc_ms",
        "r_amplitude_mv", "rr_cv"]
print("Regenerated table:", new.shape)
print("Animals:", len(new), "| clean (OK & rr_cv<=0.15):",
      int(((new.status=="OK") & (new.rr_cv<=0.15)).sum()))
print("\nGroup x study cross-tab:")
print(pd.crosstab(new["group"], new["study"]))
print("\nCritical columns present:", all(col in new.columns for col in CRIT))


## 1.10 Mains-removal filter validation

*Least-squares sinusoidal subtraction vs notch: R-amplitude preservation.*  
<sub>source: `sinusoidal_mains.py`</sub>

In [ ]:
"""Sinusoidal (regression) mains removal, and a head-to-head test against the
current Q=60 notch.

Mains interference is a stationary tone at a precise frequency (50 Hz in Ireland,
plus the 100 Hz harmonic). Rather than notching out a frequency BAND -- which also
removes the broadband QRS energy that happens to sit at 50 Hz -- we estimate the
amplitude and phase of the mains sinusoid by least squares over the whole segment
and subtract exactly that one sinusoid. Cardiac signal at 50 Hz is untouched.

For a segment x sampled at fs, and a target frequency f, the mains component is

    m(t) = A*cos(2*pi*f*t) + B*sin(2*pi*f*t)

A and B are found by projecting x onto cos and sin (ordinary least squares on the
two-column design matrix). Subtracting A*cos + B*sin removes only that tone.

To track slow drift in mains amplitude/phase over a long recording, the estimate
is done in overlapping windows and the fitted sinusoid subtracted per window
(a simple, robust version of the "spectrum-fit"/Zapline idea).

This module is import-safe: `remove_mains(x, fs)` is the drop-in replacement for
the notch loop inside bandpass_filter.
"""
import numpy as np
from scipy.signal import butter, filtfilt


def _fit_subtract_tone(x, fs, f, t0=0.0):
    """Least-squares subtract a single sinusoid of frequency f from x."""
    n = len(x)
    t = (np.arange(n) + t0) / fs
    c = np.cos(2 * np.pi * f * t)
    s = np.sin(2 * np.pi * f * t)
    # design matrix [cos, sin]; solve for [A, B]
    D = np.column_stack([c, s])
    coef, *_ = np.linalg.lstsq(D, x, rcond=None)
    return x - D @ coef, coef


def remove_mains(x, fs, freqs=(50.0, 100.0), win_s=1.0, overlap=0.5):
    """Subtract mains tones by windowed sinusoidal regression.

    win_s   : window length in seconds over which amplitude/phase are assumed
              constant (1 s tracks slow drift while keeping many mains cycles).
    overlap : fractional overlap between windows; results are cross-faded so
              there is no discontinuity at window edges.
    """
    x = np.asarray(x, float)
    n = len(x)
    w = int(round(win_s * fs))
    if w >= n:                                   # short segment: single fit
        y = x.copy()
        for f in freqs:
            y, _ = _fit_subtract_tone(y, fs, f)
        return y
    step = max(1, int(round(w * (1 - overlap))))
    y = np.zeros(n)
    wsum = np.zeros(n)
    taper = np.hanning(w)                         # cross-fade weight
    for start in range(0, n - 1, step):
        end = min(start + w, n)
        seg = x[start:end]
        tap = taper[:end - start] if end - start < w else taper
        cleaned = seg.copy()
        for f in freqs:
            cleaned, _ = _fit_subtract_tone(cleaned, fs, f, t0=start)
        y[start:end] += cleaned * tap
        wsum[start:end] += tap
    wsum[wsum == 0] = 1.0
    return y / wsum


# ---- filter variants for the head-to-head test ------------------------------
def _bandpass(x, fs, low=0.5, high=150.0, order=2):
    b, a = butter(order, [low / (fs / 2), high / (fs / 2)], btype="band")
    return filtfilt(b, a, x)


def filt_notch(x, fs):
    """Current pipeline: bandpass + Q=60 notch at 50/100."""
    from scipy.signal import iirnotch
    y = _bandpass(x, fs)
    for f0 in (50, 100):
        bn, an = iirnotch(f0 / (fs / 2), Q=60)
        y = filtfilt(bn, an, y)
    return y


def filt_sinusoidal(x, fs):
    """Proposed: bandpass, then sinusoidal subtraction of the mains tones."""
    y = _bandpass(x, fs)
    return remove_mains(y, fs, freqs=(50.0, 100.0))


def _test():
    """Head-to-head on the cohort: mains rejection and R-amplitude preservation."""
    import sys, os, pandas as pd
    # (portable) make qt_cohort_audit importable whether run as a script or a notebook cell
    try:
        _here = os.path.dirname(os.path.abspath(__file__))
    except NameError:
        _here = os.getcwd()
    sys.path.insert(0, _here)
    from qt_cohort_audit import load_pipeline_ns
    ns = load_pipeline_ns(); g = ns.__getitem__
    FS = g("FS")
    parse = g("parse_header_and_layout"); load = g("load_ecg_file")
    findbw = g("find_baseline_window"); detect = g("detect_r_peaks"); avg = g("average_beats")

    def bandpass_only(x, fs):
        return _bandpass(x, fs)

    def rr_amp(f, seg):
        fi = f(seg, FS); pk, inv, hr, _ = detect(fi)
        if inv:
            fi = -fi
        t, tm, b = avg(fi, pk)
        w = (tm >= -15) & (tm <= 35)
        return float(np.max(t[w]) - np.min(t[w]))

    def mains_power(sig, fs):
        from scipy.signal import welch
        fr, P = welch(sig - np.mean(sig), fs=fs, nperseg=4096)
        m = (fr >= 49) & (fr <= 51)
        return float(np.trapezoid(P[m], fr[m]) if hasattr(np, "trapezoid")
                     else np.trapz(P[m], fr[m]))

    m = pd.read_csv("../outputs/_metadata_merged.csv")
    m = m[(m.status == "OK") & (m.rr_cv <= 0.15)]
    loss_notch, loss_sin, rej_notch, rej_sin = [], [], [], []
    for _, r in m.iterrows():
        try:
            p = g("DATA_DIR") / str(r["source_file"])
            nh, l, c, _ = parse(p); v, mk = load(p, nh, c)
            s, e = findbw(mk, len(v))[:2]
            seg = v[s:e]
            r_bp = rr_amp(bandpass_only, seg)          # reference (no mains removal)
            r_no = rr_amp(filt_notch, seg)
            r_si = rr_amp(filt_sinusoidal, seg)
            loss_notch.append(100 * (r_bp - r_no) / r_bp)
            loss_sin.append(100 * (r_bp - r_si) / r_bp)
            raw_m = mains_power(seg, FS)
            if raw_m > 0:
                rej_notch.append(100 * (1 - mains_power(filt_notch(seg, FS), FS) / raw_m))
                rej_sin.append(100 * (1 - mains_power(filt_sinusoidal(seg, FS), FS) / raw_m))
        except Exception:
            pass
    loss_notch = np.array(loss_notch); loss_sin = np.array(loss_sin)
    print(f"n = {len(loss_notch)} clean recordings\n")
    print(f"{'':22}{'NOTCH (Q60)':>14}{'SINUSOIDAL':>14}")
    print(f"{'R-amp signal loss':22}{loss_notch.mean():>12.2f}% {loss_sin.mean():>12.2f}%")
    print(f"{'  (max)':22}{loss_notch.max():>12.2f}% {loss_sin.max():>12.2f}%")
    print(f"{'50 Hz mains removed':22}{np.mean(rej_notch):>12.1f}% {np.mean(rej_sin):>12.1f}%")
    print(f"\nsinusoidal keeps {loss_notch.mean() - loss_sin.mean():.2f} percentage points "
          f"more signal, with comparable mains rejection.")


if __name__ == "__main__":
    _test()

## 1.11 Detector robustness (MAD vs prominence)

*Two independent detectors → detector_robustness.csv + Bland–Altman agreement (used by NB2 Phase 9).*  
<sub>source: `detector_robustness.py`</sub>

In [ ]:
"""Alternative-detector robustness check (framework layer 1).
The pipeline uses a MAD-adaptive threshold detector. Here we run a DIFFERENT-principle
detector (scipy find_peaks, prominence + refractory distance) on the same filtered
signals and compare: heart-rate agreement and beat-position match rate. If the two
independent detectors agree, the extracted rate/beats are not an artefact of the MAD rule.
"""
import numpy as np, pandas as pd
from scipy.signal import find_peaks
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
from qt_cohort_audit import load_pipeline_ns, animal_id_from_name

ns = load_pipeline_ns(); g = ns.__getitem__
FS = g("FS")
parse = g("parse_header_and_layout"); load = g("load_ecg_file")
findbw = g("find_baseline_window"); bp = g("bandpass_filter"); detect = g("detect_r_peaks")
DATA_DIR = g("DATA_DIR")

m = pd.read_csv("../outputs/_metadata_merged.csv")
clean = m[(m.status == "OK") & (m.rr_cv <= 0.15)]
byid = {animal_id_from_name(p.stem): p for p in sorted(DATA_DIR.glob("*.txt"))}
REFRACT = int(0.055 * FS)
TOL = int(0.010 * FS)   # 10 ms match tolerance


def alt_detector(sig):
    """Different principle: prominence-based find_peaks with refractory distance."""
    prom = 0.4 * (np.percentile(sig, 99) - np.median(sig))
    pk, _ = find_peaks(sig, distance=REFRACT, prominence=max(prom, 1e-6))
    return pk


rows = []
for aid in clean.animal_id:
    path = byid.get(int(aid))
    if path is None:
        continue
    try:
        nh, ld, ec, ti = parse(path); volt, mk = load(path, nh, ec)
        s, e, *_ = findbw(mk, len(volt))
        if s is None:
            continue
        filt = bp(volt[s:e])
        mad_pk, inv, hr_mad, _ = detect(filt)
        if inv:
            filt = -filt
        alt_pk = alt_detector(filt)
        if len(mad_pk) < 5 or len(alt_pk) < 5:
            continue
        hr_alt = 60000.0 / (np.median(np.diff(alt_pk)) / FS * 1000.0)
        # match rate: fraction of MAD peaks with an alt peak within TOL
        matched = sum(np.any(np.abs(alt_pk - p) <= TOL) for p in mad_pk)
        rows.append(dict(aid=int(aid), hr_mad=hr_mad, hr_alt=hr_alt,
                         n_mad=len(mad_pk), n_alt=len(alt_pk),
                         match=100.0 * matched / len(mad_pk)))
    except Exception:
        continue

df = pd.DataFrame(rows)
df.to_csv("../outputs/detector_robustness.csv", index=False)
r = np.corrcoef(df.hr_mad, df.hr_alt)[0, 1]
print("Alternative-detector robustness  (n = %d clean recordings)\n" % len(df))
print("Heart-rate agreement (MAD vs prominence detector):")
print("  Pearson r = %.4f" % r)
print("  mean |HR difference| = %.1f bpm" % np.mean(np.abs(df.hr_mad - df.hr_alt)))
print("  median beat-position match rate = %.1f%% (within 10 ms)" % df.match.median())
print("  recordings with >95%% match = %d / %d" % ((df.match > 95).sum(), len(df)))

fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 5.3))
lim = [df[["hr_mad", "hr_alt"]].min().min()-20, df[["hr_mad", "hr_alt"]].max().max()+20]
a1.plot(lim, lim, color="#c0392b", lw=1.5, ls="--", label="perfect agreement")
a1.scatter(df.hr_mad, df.hr_alt, s=30, color="#1f6f8f", edgecolor="k", linewidth=0.3, alpha=0.75)
a1.set_xlim(lim); a1.set_ylim(lim)
a1.set_xlabel("heart rate — MAD detector (bpm)"); a1.set_ylabel("heart rate — prominence detector (bpm)")
a1.set_title("Heart-rate agreement between the two detectors\nPearson r = %.4f" % r,
             fontweight="bold", fontsize=11); a1.legend(fontsize=9); a1.grid(alpha=.2)
a2.hist(df.match, bins=20, color="#aed6e6", edgecolor="k")
a2.axvline(95, color="#c0392b", lw=1.5, ls="--", label="95% match")
a2.set_xlabel("beat-position match rate (%)"); a2.set_ylabel("recordings")
a2.set_title("Beat-position agreement (within 10 ms)\nmedian %.1f%%" % df.match.median(),
             fontweight="bold", fontsize=11); a2.legend(fontsize=9); a2.grid(alpha=.2)
fig.suptitle("R-peak detector robustness — MAD-adaptive vs an independent prominence-based detector",
             fontweight="bold", fontsize=12.5)
fig.tight_layout(rect=[0, 0, 1, 0.93])
fig.savefig("../outputs/figures/detector_robustness.png", dpi=145, bbox_inches="tight")
print("\nsaved detector_robustness.png")

## 1.12 Rich per-beat feature set

*Enriched morphology features → rich_features.csv (used by NB2 grouping).*  
<sub>source: `rich_features_grouping.py`</sub>

In [ ]:
"""Last grouping attempt: richer features than the standard landmarks.

Extracts, per clean animal:
  - HRV metrics from the RR series: SDNN, RMSSD, pNN (fast-mouse analogue),
    CV, and a Poincare SD1/SD2
  - whole-beat shape: the averaged beat is length-normalised and reduced by
    functional PCA (the first few FPCA scores capture morphology the scalar
    landmarks miss)

Then repeats the full grouping battery on this enriched matrix:
  - one-way ANOVA per feature (group), per study
  - supervised LDA classification (5-group and Control-vs-Dox) with permutation
  - unsupervised k=5 clustering vs true labels (adjusted Rand index)

If grouping still fails here, it fails on every reasonable feature set.
"""
import warnings
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score

warnings.filterwarnings("ignore")
import sys, os
# (portable) make qt_cohort_audit importable whether run as a script or a notebook cell
try:
    _here = os.path.dirname(os.path.abspath(__file__))
except NameError:
    _here = os.getcwd()
sys.path.insert(0, _here)
from qt_cohort_audit import load_pipeline_ns


def hrv(rr_ms):
    rr = np.asarray(rr_ms, float)
    if len(rr) < 5:
        return dict(sdnn=np.nan, rmssd=np.nan, cv=np.nan, sd1=np.nan, sd2=np.nan)
    d = np.diff(rr)
    sdnn = float(np.std(rr, ddof=1))
    rmssd = float(np.sqrt(np.mean(d ** 2)))
    sd1 = float(np.sqrt(0.5) * np.std(d, ddof=1))
    sd2 = float(np.sqrt(max(2 * sdnn ** 2 - 0.5 * np.std(d, ddof=1) ** 2, 0)))
    return dict(sdnn=sdnn, rmssd=rmssd, cv=sdnn / np.mean(rr), sd1=sd1, sd2=sd2)


def main():
    ns = load_pipeline_ns(); g = ns.__getitem__
    FS = g("FS")
    parse = g("parse_header_and_layout"); load = g("load_ecg_file")
    findbw = g("find_baseline_window"); bp = g("bandpass_filter")
    detect = g("detect_r_peaks"); avg = g("average_beats")

    meta = pd.read_csv("../outputs/_metadata_merged.csv")
    meta = meta[(meta.status == "OK") & (meta.rr_cv <= 0.15)].copy()

    templates, rows = [], []
    for _, r in meta.iterrows():
        try:
            p = g("DATA_DIR") / str(r["source_file"])
            nh, l, c, _ = parse(p); v, mk = load(p, nh, c)
            s, e = findbw(mk, len(v))[:2]
            filt = bp(v[s:e]); pk, inv, hrb, _ = detect(filt)
            if inv:
                filt = -filt
            rr = np.diff(pk) * (1000.0 / FS)
            tmpl, t_ms, beats = avg(filt, pk)
            base = np.median(tmpl[t_ms < -70]) if (t_ms < -70).any() else 0.0
            row = dict(animal_id=r["animal_id"], group=r["group"], study=r["study"])
            row.update(hrv(rr))
            rows.append(row)
            templates.append(tmpl - base)
        except Exception:
            pass

    df = pd.DataFrame(rows)
    T = np.vstack(templates)                        # whole-beat matrix
    # functional PCA of the beat shape
    fpca = PCA(n_components=5).fit(StandardScaler(with_std=False).fit_transform(T))
    scores = fpca.transform(StandardScaler(with_std=False).fit_transform(T))
    for i in range(5):
        df[f"fpca{i+1}"] = scores[:, i]
    print(f"n = {len(df)} | FPCA variance explained: "
          f"{np.round(fpca.explained_variance_ratio_[:5], 3)}")

    RICH = ["sdnn", "rmssd", "cv", "sd1", "sd2", "fpca1", "fpca2", "fpca3", "fpca4", "fpca5"]
    for c in RICH:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df[RICH] = df[RICH].fillna(df[RICH].median())

    for study in ["acute", "chronic"]:
        d = df[df.study == study]
        print(f"\n########## {study.upper()}  (n={len(d)}) ##########")
        # 1) one-way ANOVA per rich feature
        print("  one-way ANOVA (group) on rich features:")
        anyp = False
        for c in RICH:
            groups = [d[d.group == gp][c].values for gp in d.group.unique()]
            groups = [x for x in groups if len(x) >= 2]
            if len(groups) < 2:
                continue
            F, p = stats.f_oneway(*groups)
            flag = " <-- p<0.05" if p < 0.05 else ""
            if p < 0.05:
                anyp = True
            print(f"     {c:8s} p={p:.3f}{flag}")
        if not anyp:
            print("     -> nothing significant")
        # 2) supervised LDA, 5-group
        X = d[RICH].values; y = d["group"].values
        cc = np.unique(y, return_counts=True)[1]
        if cc.min() >= 2:
            k = int(min(5, cc.min()))
            cv = StratifiedKFold(k, shuffle=True, random_state=0)
            pipe = make_pipeline(StandardScaler(), LinearDiscriminantAnalysis())
            acc = cross_val_score(pipe, X, y, cv=cv, scoring="balanced_accuracy").mean()
            rng = np.random.default_rng(0)
            null = [cross_val_score(pipe, X, rng.permutation(y), cv=cv,
                                    scoring="balanced_accuracy").mean() for _ in range(200)]
            pp = (np.sum(np.array(null) >= acc) + 1) / 201
            print(f"  LDA 5-group: acc {acc:.3f} (chance 0.20, null {np.mean(null):.3f}) "
                  f"p={pp:.3f} -> {'SIGNIFICANT' if pp < 0.05 else 'at chance'}")
        # 3) unsupervised k=5 vs truth
        Xs = StandardScaler().fit_transform(X)
        lab = KMeans(5, n_init=10, random_state=0).fit_predict(Xs)
        ari = adjusted_rand_score(y, lab)
        print(f"  unsupervised k=5 vs true groups: ARI {ari:+.3f} "
              f"-> {'structure' if ari > 0.2 else 'random'}")

    df.to_csv("../outputs/rich_features.csv", index=False)
    print("\nsaved ../outputs/rich_features.csv")


if __name__ == "__main__":
    main()